# 04 - 提取 10-K 核心章节

这一节把完整财报文本转换成结构化章节。我们先提取三个投研中常用的部分：

- `Item 1. Business`：公司业务
- `Item 1A. Risk Factors`：风险因素
- `Item 7. MD&A`：管理层讨论与分析

这里不调用大模型。章节边界由可检查、可重复运行的 Python 规则确定。

## 1. 读取上一节生成的纯文本

Notebook 会自动读取指定公司目录中最新的 `10-K` 文本文件。

In [ ]:
import json
import re
from pathlib import Path

TICKER = "AAPL"
FORM_TYPE = "10-K"
DATA_DIR = Path("data/sec")

text_files = sorted(
    (DATA_DIR / TICKER).glob(f"*_{FORM_TYPE}_*.txt"),
    reverse=True,
)
if not text_files:
    raise FileNotFoundError(
        f"没有找到 {TICKER} 的 {FORM_TYPE} 纯文本，请先运行 03_parse_filing.ipynb"
    )

text_path = text_files[0]
filing_text = text_path.read_text(encoding="utf-8")

print(f"读取文件：{text_path}")
print(f"总字符数：{len(filing_text):,}")

## 2. 定义标准章节规则

每个章节包含开始标题、可能的结束标题和最小合理长度。备用结束标题用于兼容部分公司省略某个 Item 的情况。

In [ ]:
ITEM_TITLES = {
    "1": ("business",),
    "1A": ("risk factors",),
    "1B": ("unresolved staff comments",),
    "1C": ("cybersecurity",),
    "2": ("properties",),
    "7": ("management", "discussion"),
    "7A": ("quantitative", "market risk"),
    "8": ("financial statements",),
}

SECTION_SPECS = {
    "business": {
        "title": "Item 1. Business",
        "start_item": "1",
        "end_items": ("1A",),
        "min_chars": 1000,
    },
    "risk_factors": {
        "title": "Item 1A. Risk Factors",
        "start_item": "1A",
        "end_items": ("1B", "1C", "2"),
        "min_chars": 1000,
    },
    "mda": {
        "title": "Item 7. Management's Discussion and Analysis",
        "start_item": "7",
        "end_items": ("7A", "8"),
        "min_chars": 1000,
    },
}

SECTION_SPECS

## 3. 查找标题候选位置

同一个标题通常会在目录和正文各出现一次。候选函数要求一行以 `Item 编号` 开头，并检查标题本身或下一行是否包含预期标题词，从而排除普通句子中的 Item 引用。

In [ ]:
def find_item_candidates(text: str, item_number: str) -> list[dict]:
    pattern = re.compile(
        rf"(?im)^[ \t]*item[ \t]+{re.escape(item_number)}"
        rf"(?![a-z0-9])[ \t]*(?:[.\-:—])?[^\n]*"
    )
    expected_words = ITEM_TITLES[item_number]
    candidates = []

    for match in pattern.finditer(text):
        heading_line = match.group(0).strip()
        prefix_pattern = (
            rf"(?i)^item[ \t]+{re.escape(item_number)}"
            rf"(?![a-z0-9])[ \t]*(?:[.\-:—])?[ \t]*"
        )
        heading_tail = re.sub(prefix_pattern, "", heading_line).strip()

        following_lines = [
            line.strip()
            for line in text[match.end():match.end() + 250].splitlines()
            if line.strip()
        ]
        next_line = following_lines[0] if following_lines else ""
        title_context = f"{heading_tail} {next_line}".lower()

        if all(word in title_context for word in expected_words):
            candidates.append({
                "item": item_number,
                "start": match.start(),
                "end": match.end(),
                "heading": heading_line,
                "next_line": next_line,
            })

    return candidates

In [ ]:
for item_number in ("1", "1A", "1B", "1C", "2", "7", "7A", "8"):
    candidates = find_item_candidates(filing_text, item_number)
    positions = [candidate["start"] for candidate in candidates]
    print(f"Item {item_number:>2}: {len(candidates)} 个候选，位置 {positions}")

## 4. 根据相邻标题提取正文

对每一个开始候选，我们寻找它后面最近的结束章节标题。目录中的两个标题距离通常很短，正文标题之间距离较长，因此选择长度最大的有效区间，并执行最小长度校验。

In [ ]:
def extract_section(text: str, section_name: str, spec: dict) -> dict:
    start_candidates = find_item_candidates(text, spec["start_item"])
    end_candidates = []

    for end_item in spec["end_items"]:
        end_candidates.extend(find_item_candidates(text, end_item))

    possible_ranges = []
    for start in start_candidates:
        later_ends = [
            end for end in end_candidates if end["start"] > start["start"]
        ]
        if not later_ends:
            continue

        nearest_end = min(later_ends, key=lambda candidate: candidate["start"])
        possible_ranges.append({
            "start": start,
            "end": nearest_end,
            "char_count": nearest_end["start"] - start["start"],
        })

    if not possible_ranges:
        raise ValueError(f"无法确定 {spec['title']} 的起止位置")

    selected = max(possible_ranges, key=lambda item: item["char_count"])
    if selected["char_count"] < spec["min_chars"]:
        raise ValueError(
            f"{spec['title']} 只有 {selected['char_count']} 个字符，"
            "可能错误地选择了目录标题"
        )

    start_position = selected["start"]["start"]
    end_position = selected["end"]["start"]
    section_text = text[start_position:end_position].strip()

    return {
        "name": section_name,
        "item": spec["start_item"],
        "title": spec["title"],
        "start": start_position,
        "end": end_position,
        "char_count": len(section_text),
        "end_item": selected["end"]["item"],
        "text": section_text,
    }


def extract_10k_sections(text: str) -> dict[str, dict]:
    return {
        name: extract_section(text, name, spec)
        for name, spec in SECTION_SPECS.items()
    }

In [ ]:
sections = extract_10k_sections(filing_text)

for name, section in sections.items():
    print(
        f"{name:>12}: {section['char_count']:>7,} 字符 | "
        f"位置 {section['start']:,} - {section['end']:,} | "
        f"结束于 Item {section['end_item']}"
    )

## 5. 人工检查提取结果

自动规则必须配合可观察的校验。下面分别显示每个章节的开头和结尾，确认没有从目录开始，也没有越过下一章节。

In [ ]:
def preview_section(section: dict, preview_chars: int = 500) -> None:
    print("=" * 80)
    print(section["title"])
    print(f"字符数：{section['char_count']:,}")
    print("\n[开头]")
    print(section["text"][:preview_chars])
    print("\n[结尾]")
    print(section["text"][-preview_chars:])


for section in sections.values():
    preview_section(section)

## 6. 执行结构校验

除了检查长度，还要确认章节按照 10-K 的标准顺序出现，并且正文中包含预期关键词。

In [ ]:
expected_order = ("business", "risk_factors", "mda")
positions = [sections[name]["start"] for name in expected_order]
assert positions == sorted(positions), "章节顺序异常"

assert "business" in sections["business"]["text"][:300].lower()
assert "risk factors" in sections["risk_factors"]["text"][:300].lower()
assert "management" in sections["mda"]["text"][:300].lower()

print("章节顺序、长度和标题关键词检查通过")

## 7. 保存标准化 JSON

JSON 同时保存章节正文和它在完整文本中的字符位置。后续引用原文时，可以用 `start` 和 `end` 追溯来源。

In [ ]:
result = {
    "ticker": TICKER,
    "form": FORM_TYPE,
    "source_file": str(text_path),
    "sections": sections,
}

output_path = text_path.with_name(f"{text_path.stem}_sections.json")
output_path.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"章节文件已保存：{output_path.resolve()}")

## 这一节完成了什么

现在完整财报已经被转换成统一的章节结构，并保留了原文位置。下一步可以把每个章节切成适合模型处理的小块，再使用 DeepSeek 分别分析业务、风险和管理层观点。

当前规则已在苹果 10-K 上验证，但还需要在更多公司的 10-K 上运行测试，才能评估它的通用性。